# Chapter 9.5 - Recurrent Neural Network Implementation from Scratch

This notebook implements a character-level RNN language model from explicit parameters. It covers state initialization, forward recurrence, gradient clipping, a transparent training epoch, and prefix-conditioned decoding.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- identify every trainable parameter in a scratch RNN
- trace token IDs through one-hot inputs, hidden states, and logits
- explain why carried state is detached between minibatches
- clip a global gradient norm without changing its direction
- decode a prefix and generate additional tokens


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.5.0 The Problem This Notebook Solves

Framework RNN layers hide a time loop and several matrix operations. Here we expose them. The implementation is still a PyTorch `nn.Module`, so parameters register correctly with optimizers, but the recurrent computation itself is handwritten.

`nn.Parameter` wraps a tensor that should be trainable. Assigning it as an attribute of a module registers it. `model.parameters()` can then find it. Calling `model(X, state)` invokes the module's `forward` method through `nn.Module.__call__`, which also manages framework hooks.


## 9.5.1 RNN Model: Explicit Parameters and State

Parameter shapes are contracts:

```text
W_xh: (vocab, hidden)
W_hh: (hidden, hidden)
W_hq: (hidden, vocab)
```

The input arrives as token IDs shaped `(batch, time)`. The model one-hot encodes and transposes them to `(time, batch, vocab)`. It returns flattened logits `(batch * time, vocab)` plus final state `(batch, hidden)`.


In [ ]:
class ScratchRNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, sigma=0.01):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * sigma)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * sigma)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        self.W_hq = nn.Parameter(torch.randn(hidden_size, vocab_size) * sigma)
        self.b_q = nn.Parameter(torch.zeros(vocab_size))

    def begin_state(self, batch_size, device=None):
        return torch.zeros(batch_size, self.hidden_size, device=device)

    def forward(self, token_ids, state):
        inputs = F.one_hot(token_ids.T, self.vocab_size).float()
        outputs = []
        for X_t in inputs:
            state = torch.tanh(X_t @ self.W_xh + state @ self.W_hh + self.b_h)
            outputs.append(state @ self.W_hq + self.b_q)
        return torch.cat(outputs, dim=0), state


In [ ]:
model = ScratchRNNLM(vocab_size=7, hidden_size=8)
X = torch.tensor([[0, 1, 2, 3], [3, 2, 1, 0]])
state = model.begin_state(batch_size=2, device=X.device)
logits, next_state = model(X, state)

print("registered parameters:", [name for name, _ in model.named_parameters()])
print("logits:", shape(logits), "state:", shape(next_state))
assert shape(logits) == (8, 7)
assert shape(next_state) == (2, 8)
assert len(list(model.parameters())) == 5


## 9.5.2 RNN-Based Language Model Loss and Axis Alignment

The model concatenates outputs in time-major order: all batch examples at time 0, then all at time 1, and so on. Therefore targets must transpose from `(batch, time)` to `(time, batch)` before flattening.


In [ ]:
Y = torch.tensor([[1, 2, 3, 4], [2, 1, 0, 6]])
targets = Y.T.reshape(-1)
loss = F.cross_entropy(logits, targets)

print("targets in model output order:", targets)
print("average token loss:", loss.item())
assert shape(targets) == (8,)
assert targets[:2].tolist() == [1, 2]
assert loss.ndim == 0


## 9.5.3 Gradient Clipping

The **global gradient norm** treats all parameter gradients as pieces of one long vector. If its length exceeds threshold `theta`, clipping multiplies every gradient by the same factor `theta / norm`. The direction is preserved while the step magnitude is capped.

Clipping handles unusually large gradients; it does not repair bad data, guarantee convergence, or recover a gradient that has already vanished.


In [ ]:
def clip_gradients(module, theta):
    gradients = [p.grad for p in module.parameters() if p.grad is not None]
    if not gradients:
        return 0.0
    norm = torch.sqrt(sum(torch.sum(gradient ** 2) for gradient in gradients))
    if norm > theta:
        scale = theta / (norm + 1e-12)
        for gradient in gradients:
            gradient.mul_(scale)
    return norm.item()

model.zero_grad()
loss.backward()
norm_before = clip_gradients(model, theta=0.25)
norm_after = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters())).item()
print({"before": norm_before, "after": norm_after})
assert norm_after <= 0.25001


## 9.5.4 Training With Consecutive Minibatches

This inline corpus repeats a short pattern. `sequence_batches` makes contiguous `(batch, time)` minibatches. At the start of an epoch, state is zeros. Between adjacent minibatches, the numeric state is carried forward but detached from its old computation graph.

Detaching says: use these state values as the next starting point, but do not backpropagate through all earlier minibatches. This is **truncated backpropagation through time** at minibatch boundaries.


In [ ]:
def sequence_batches(corpus, batch_size, num_steps):
    usable = ((len(corpus) - 1) // batch_size) * batch_size
    Xs = corpus[:usable].reshape(batch_size, -1)
    Ys = corpus[1 : usable + 1].reshape(batch_size, -1)
    for start in range(0, Xs.shape[1] - num_steps + 1, num_steps):
        yield Xs[:, start : start + num_steps], Ys[:, start : start + num_steps]

corpus = torch.tensor(([0, 1, 2, 3, 4, 5, 6] * 30), dtype=torch.long)
model = ScratchRNNLM(vocab_size=7, hidden_size=16)
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

def train_epoch(model, corpus, batch_size=2, num_steps=7):
    state = model.begin_state(batch_size, corpus.device)
    total_loss = 0.0
    token_count = 0
    for X, Y in sequence_batches(corpus, batch_size, num_steps):
        state = state.detach()
        optimizer.zero_grad()
        logits, state = model(X, state)
        targets = Y.T.reshape(-1)
        loss = F.cross_entropy(logits, targets)
        loss.backward()
        clip_gradients(model, theta=1.0)
        optimizer.step()
        total_loss += loss.item() * targets.numel()
        token_count += targets.numel()
    return math.exp(total_loss / token_count)

perplexities = [train_epoch(model, corpus) for _ in range(8)]
print(perplexities)
assert all(math.isfinite(value) for value in perplexities)


## 9.5.5 Decoding: Warm Up With a Prefix, Then Generate

**Decoding** converts model scores into output tokens. Greedy decoding selects the largest logit at each step. The prefix first warms the hidden state using known tokens. After the prefix ends, each predicted ID becomes the next input.

Greedy generation is deterministic and easy to inspect, but it is not the only strategy. Sampling and beam search make different quality-diversity tradeoffs.


In [ ]:
def predict(prefix, num_predictions, model):
    state = model.begin_state(batch_size=1)
    outputs = [prefix[0]]
    for token in prefix[1:]:
        X = torch.tensor([[outputs[-1]]])
        _, state = model(X, state)
        outputs.append(token)
    for _ in range(num_predictions):
        X = torch.tensor([[outputs[-1]]])
        logits, state = model(X, state)
        outputs.append(int(logits[-1].argmax()))
    return outputs

generated = predict(prefix=[0, 1], num_predictions=10, model=model)
print(generated)
assert generated[:2] == [0, 1]
assert len(generated) == 12
assert all(0 <= token < model.vocab_size for token in generated)


## 9.5.6 Break It Deliberately: Reuse Attached State

After `backward()`, PyTorch normally frees intermediate graph buffers. Reusing a state still attached to that graph and calling `backward()` again tries to traverse freed history. Detach the carried state at the minibatch boundary.


In [ ]:
probe = ScratchRNNLM(vocab_size=7, hidden_size=4)
state = probe.begin_state(batch_size=1)
X1 = torch.tensor([[0, 1]])
Y1 = torch.tensor([1, 2])
logits, state = probe(X1, state)
F.cross_entropy(logits, Y1).backward()

try:
    logits2, state = probe(torch.tensor([[2, 3]]), state)
    F.cross_entropy(logits2, torch.tensor([3, 4])).backward()
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected backward through freed history to fail")


## 9.5 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. Why does assigning nn.Parameter attributes register trainable tensors?
2. Why are labels transposed before flattening?
3. How does global-norm clipping change gradient magnitude and direction?
4. What information is kept and cut by state.detach()?
5. What happens during prefix warm-up in decoding?
